In [ ]:
from getpass import getpass
import sys, os

REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"
LOCAL_REPO_PATH = f"/content/{REPO_NAME}"

# Mount Google Drive first (Colab only)
if '/content' in os.getcwd():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

# Clone only if not existing
if not os.path.exists(LOCAL_REPO_PATH):
    print("Cloning repo...")
    token = getpass("GitHub Token: ")
    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir("/content")

# Enable import of your modules
if LOCAL_REPO_PATH not in sys.path:
    os.chdir(LOCAL_REPO_PATH)
    sys.path.append(os.getcwd())
    os.chdir("/content")

In [ ]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sps

from Challenge import paths
importlib.reload(paths)

In [ ]:
# Ids required for the submission
user_ids_test = pd.read_csv(paths.CHALLENGE_USER_IDS_TEST)
print("Number of users in the test set: {}".format(len(user_ids_test)))
user_ids_test.head(n=10)

In [ ]:
# Import data
URM_all_dataframe = pd.read_csv(filepath_or_buffer=paths.CHALLENGE_DATASET,
                                dtype={0:int, 1:int, 2:float, 3:int})

print ("The number of interactions is {}".format(len(URM_all_dataframe)))
URM_all_dataframe.head(n=10)

In [ ]:
# Extract the list of unique user id and item id
userID_unique = URM_all_dataframe["UserID"].unique()
itemID_unique = URM_all_dataframe["ItemID"].unique()

n_users = len(userID_unique)
n_items = len(itemID_unique)
n_interactions = len(URM_all_dataframe)

print ("Number of items\t {}, Number of users\t {}".format(n_items, n_users))
print ("Max ID items\t {}, Max Id users\t {}\n".format(max(itemID_unique), max(userID_unique)))

In [ ]:
# Remove empty indices with a new mapping
mapped_id, original_id = pd.factorize(URM_all_dataframe["UserID"].unique())
user_original_ID_to_index = pd.Series(mapped_id, index=original_id)

mapped_id, original_id = pd.factorize(URM_all_dataframe["ItemID"].unique())
item_original_ID_to_index = pd.Series(mapped_id, index=original_id)

# Check test users are in the mapping
missing_test_users = np.setdiff1d(user_ids_test.values.flatten(), user_original_ID_to_index.index.values)
print("Number of missing test users: {}".format(len(missing_test_users)))
assert len(missing_test_users) == 0, "There are test users without interactions in the dataset!"

In [ ]:
# Replace the IDs in the URM dataframe
URM_all_dataframe["UserID"] = URM_all_dataframe["UserID"].map(user_original_ID_to_index)
URM_all_dataframe["ItemID"] = URM_all_dataframe["ItemID"].map(item_original_ID_to_index)
URM_all_dataframe.head(n=10)

In [ ]:
# Replace test user ids with the new mapped ids
user_ids_test["UserID"] = user_ids_test["UserID"].map(user_original_ID_to_index)
user_ids_test.head(n=10)

In [ ]:
# Check new number of users and items after re-mapping
userID_unique = URM_all_dataframe["UserID"].unique()
itemID_unique = URM_all_dataframe["ItemID"].unique()

n_users = len(userID_unique)
n_items = len(itemID_unique)
n_interactions = len(URM_all_dataframe)

print ("Number of items\t {}, Number of users\t {}".format(n_items, n_users))
print ("Max ID items\t {}, Max Id users\t {}\n".format(max(itemID_unique), max(userID_unique)))
print ("Average interactions per user {:.2f}".format(n_interactions/n_users))
print ("Average interactions per item {:.2f}\n".format(n_interactions/n_items))

print ("Sparsity {:.2f} %".format((1-float(n_interactions)/(n_items*n_users))*100))

In [ ]:
# Create the User-Rating-Matrix in sparse format
URM_all = sps.coo_matrix((URM_all_dataframe["Interaction"].values, 
                          (URM_all_dataframe["UserID"].values, URM_all_dataframe["ItemID"].values)))

URM_all = URM_all.tocsr()

In [ ]:
# Plot item popularity
item_popularity = np.ediff1d(URM_all.tocsc().indptr)
item_popularity = np.sort(item_popularity)

plt.plot(item_popularity, 'ro')
plt.ylabel('Num Interactions ')
plt.xlabel('Sorted Item')
plt.show()

In [ ]:
ten_percent = int(n_items/10)

print("Average per-item interactions over the whole dataset {:.2f}".
      format(item_popularity.mean()))

print("Average per-item interactions for the top 10% popular items {:.2f}".
      format(item_popularity[-ten_percent:].mean()))

print("Average per-item interactions for the least 10% popular items {:.2f}".
      format(item_popularity[:ten_percent].mean()))

print("Average per-item interactions for the median 10% popular items {:.2f}".
      format(item_popularity[int(n_items*0.45):int(n_items*0.55)].mean()))

# Check for items with no interactions
print("Number of items with zero interactions {}".
      format(np.sum(item_popularity==0)))

In [ ]:
# Plot user activity
user_activity = np.ediff1d(URM_all.tocsr().indptr)
user_activity = np.sort(user_activity)

plt.plot(user_activity, 'ro')
plt.ylabel('Num Interactions ')
plt.xlabel('Sorted User')
plt.show()

In [ ]:
# Save the preprocessed URM, the mappings and the test user ids
sps.save_npz(paths.URM_PATH, URM_all)
user_original_ID_to_index.to_csv(paths.USER_MAPPING, index_label="OriginalID", header=["MappedIndex"])
item_original_ID_to_index.to_csv(paths.ITEM_MAPPING, index_label="OriginalID", header=["MappedIndex"])
user_ids_test.to_csv(paths.TEST_USER_IDS, index=False)